In [2]:
import os, sys; from pathlib import Path
os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))

In [3]:
import numpy as np, pandas as pd
from src.config import PART2
from src.target import load_policies
D = PART2["data_dir"]; w = pd.read_parquet(D/"weather.parquet")
have = set(w.loc[w.year <= 2024, "address_id"])
for l in ["motor", "buildings", "contents"]:
    p, pt = load_policies(l, "train", D), load_policies(l, "test", D)
    nb = pt.new_business.astype(bool)
    print(f"{l}: address has 2023-24 weather — train {p.address_id.isin(have).mean():.1%} | "
          f"test renewal {pt[~nb].address_id.isin(have).mean():.1%} | test NB {pt[nb].address_id.isin(have).mean():.1%}")

motor: address has 2023-24 weather — train 100.0% | test renewal 100.0% | test NB 100.0%
buildings: address has 2023-24 weather — train 100.0% | test renewal 100.0% | test NB 100.0%
contents: address has 2023-24 weather — train 100.0% | test renewal 100.0% | test NB 100.0%


In [5]:
import copy, src.pipeline as P, src.target as T
from src.config import PART2
cfg = copy.deepcopy(PART2); T.AMOUNTS_NET = True; claims = P.load_claims(cfg["data_dir"])
show = lambda x: x.hist_1y.fillna("missing").value_counts(normalize=True).round(3).to_dict()
for l in ["motor", "buildings", "contents"]:
    d = P.build_line(l, cfg, claims); tr, te = d["tr"], d["te"]
    nb_tr, nb_te = tr.new_business.astype(bool), te.new_business.astype(bool)
    print(l, "| train 2023:", show(tr[tr.year == 2023]), "| train 2024 NB:", show(tr[(tr.year == 2024) & nb_tr]),
          "| test NB:", show(te[nb_te]), "| test renewal:", show(te[~nb_te]))

motor | train 2023: {'missing': 1.0} | train 2024 NB: {'missing': 1.0} | test NB: {'missing': 1.0} | test renewal: {'0': 0.862, '1': 0.12, '2+': 0.018}
14:00:54 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
14:00:55 buildings: fire priced separately at 97.05 per exposure (developed)
14:00:55 buildings: own GLM for ['flood']; main GLM on the other perils
buildings | train 2023: {'missing': 1.0} | train 2024 NB: {'missing': 1.0} | test NB: {'missing': 1.0} | test renewal: {'0': 0.961, '1': 0.037, '2+': 0.002}
14:00:55 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
contents | train 2023: {'missing': 1.0} | train 2024 NB: {'missing': 1.0} | test NB: {'missing': 1.0} | test renewal: {'0': 0.971, '1': 0.028, '2+': 0.001}


In [6]:
import numpy as np, pandas as pd
import src.target as T
from src.config import PART2
from src.target import load_claims, load_policies, match_claims, adjust_claims, lag_cdf_lynden_bell, VALUATION_DATE
T.AMOUNTS_NET = True; D = PART2["data_dir"]; c = load_claims(D)
for l in ["motor", "buildings", "contents"]:
    p = load_policies(l, "train", D); m = adjust_claims(match_claims(c, p, l, True, cap=None), PART2["adj"][l])
    share = m.groupby("peril").net_adj.sum(); share /= share.sum()
    F = {k: lag_cdf_lynden_bell(g.report_lag, (VALUATION_DATE - g.loss_date).dt.days) for k, g in m.groupby("peril")}
    for y in [2023, 2024]:
        days = pd.date_range(f"{y}-01-01", f"{y}-12-31"); age = (VALUATION_DATE - days).days.to_numpy()
        cday = sum(share[k] * F[k](age) for k in share.index); cum = np.concatenate([[0], np.cumsum(cday)])
        q = p[p.year == y]; s0 = (q.period_start - days[0]).dt.days.to_numpy(); e0 = (q.period_end - days[0]).dt.days.to_numpy() + 1
        rowc = (cum[e0] - cum[s0]) / (e0 - s0); ex = q.exposure_clean.to_numpy(); nb = q.new_business.astype(bool).to_numpy()
        print(f"{l} {y}: year average {cday.mean():.3f} | row-level NB {np.average(rowc[nb], weights=ex[nb]):.3f} "
              f"| renewal {np.average(rowc[~nb], weights=ex[~nb]):.3f} | NB mean exposure {ex[nb].mean():.2f}")

motor 2023: year average 0.984 | row-level NB 0.977 | renewal 0.984 | NB mean exposure 0.46
motor 2024: year average 0.839 | row-level NB 0.777 | renewal 0.847 | NB mean exposure 0.47
buildings 2023: year average 1.000 | row-level NB 0.999 | renewal 1.000 | NB mean exposure 0.48
buildings 2024: year average 0.953 | row-level NB 0.922 | renewal 0.958 | NB mean exposure 0.48
contents 2023: year average 0.998 | row-level NB 0.996 | renewal 0.998 | NB mean exposure 0.47
contents 2024: year average 0.965 | row-level NB 0.944 | renewal 0.970 | NB mean exposure 0.47


In [7]:
import pandas as pd
from src.config import PART2
from src.target import load_claims, load_policies
D = PART2["data_dir"]; w = pd.read_parquet(D/"weather.parquet"); c = load_claims(D)
p = load_policies("buildings", "train", D); pt = load_policies("buildings", "test", D)
addr = pd.concat([p.address_id, pt.address_id]).unique()
print(w[w.address_id.isin(addr)].groupby("year")[["wildfire_index", "dry_summer_index", "hail_days", "freezing_days"]].mean().round(3))
f = c[(c.line == "buildings") & (c.peril == "fire")]
print("fire claims by loss year:", f.loss_date.dt.year.value_counts().sort_index().to_dict(),
      "| exposure by year:", p.groupby("year").exposure_clean.sum().round(0).to_dict())

      wildfire_index  dry_summer_index  hail_days  freezing_days
year                                                            
2023           0.503             0.679     13.986         21.068
2024           0.549             0.711     17.133         17.520
2025           0.528             0.605     11.422         28.899
fire claims by loss year: {2023: 49, 2024: 21} | exposure by year: {2023: 11373.0, 2024: 12095.0}


In [8]:
from pathlib import Path
d = Path.home()/"IAA"/"DECISIONS.md"; s = d.read_text()
BLOCK = """### Independent review of Part 2, triaged (Oct 24) [AI 109]
- Fresh chat (Claude Opus 5.5) with config, pipeline, target, the Dataset Description and the Part 2 log.
- **Rebutted with data:** recoveries already in incurred (no: fixed shares 0.8 / 0.3, never above incurred);
  730-day truncation of completeness (emerging vs developed Lynden-Bell within 0.01-0.04); IBNER (Oct 17: open and
  closed severity alike); test-only missing weather for broker addresses (100% of test new-business addresses have
  2023-24 weather); history encoding (test new business and training 2023 / 2024 new business share the 'missing'
  level); fire as a weather effect (2023, 49 fires, was less dry than 2024, 21 fires: wildfire 0.503 vs 0.549, dry
  summer 0.679 vs 0.711; 2024 emerging fire frequency 1.74 per 1,000 matches developed 1.7-2.1, so the developed
  flat rate stands; 2025 is wetter, 0.605).
- **Confirmed:** completeness by calendar year biases new business. 2024 row-level completeness (each row's own days)
  vs the year average used: motor new business 0.777 vs 0.839 (renewals 0.847), buildings 0.922 vs 0.953, contents
  0.944 vs 0.965 (2023 within 0.01). New-business rows (mean exposure 0.47, mostly late in the year) were given too
  much weight, so the new-business risk was under-estimated by about 4% motor, 2% buildings, 1.5% contents; the fold
  could not see it (same weights on validation actuals). Fixed as E22.
- **Accepted, small:** sev_trend docstring ("gross amounts") outdated; BI trend 0% borrowed although the emerging
  market has no reform (sensitivity); buildings large-loss loading off (developed observed 2.33, about 0.7%).
- **Accepted for the report:** probe mathematics confirmed (derivation and curvature, ±0.0007 per 0.1 of load);
  report k* intervals from subset bootstrap; three classes of assumption (documented definition / calibrated on the
  leaderboard / borrowed and unverifiable), each with "would be wrong if"; premium sensitivity (±8% of the book per
  0.1 of load) next to score sensitivity; what an insurer would do in practice (broker loss bordereau, review clause,
  monthly A/E monitoring of the book).
- **Deferred:** customer snapshot variables (licence_points) as possible same-year leakage.

### E22: completeness by each row's own period (Oct 24) — rule fixed before running
- Year completeness (the total by year) unchanged; within a year each row's weight is scaled by its own period's
  completeness relative to the year's exposure-weighted average (Lynden-Bell by peril, amount shares by peril).
  `ROW_COMPLETENESS` in src/target.py (default False; Part 1 unchanged), on in PART2.
- Rule (definition correction, Oct 21): adopt if validation D² does not fall by more than 0.0005 and tails stay
  within 0.1. v5a / v5 are rebuilt with E22 before the 8 pm submissions.

"""
assert "### E22:" not in s
i = s.find("## Assumptions register"); d.write_text(s[:i] + BLOCK + s[i:]); print("inserted")

inserted


In [9]:
from pathlib import Path
R = Path.home()/"IAA"
def patch(f, pairs):
    s = f.read_text()
    for old, new in pairs:
        assert s.count(old) == 1, (f.name, old[:60]); s = s.replace(old, new)
    f.write_text(s); print("patched", f.name)
patch(R/"src"/"target.py", [
    ("AMOUNTS_NET = False\n",
     "AMOUNTS_NET = False\n# Completeness of each row's own period within its year (DECISIONS Oct 24, E22). False reproduces Part 1.\nROW_COMPLETENESS = False\n"),
    ('    df["weight"] = df["exposure_clean"] * df["year"].map(comp)\n',
     '    df["weight"] = df["exposure_clean"] * df["year"].map(comp)\n'
     '    if ROW_COMPLETENESS:\n'
     '        df["weight"] = df["weight"] * row_timing_factor(mc, policies, valuation_date)\n'),
    ("def build_model_target(",
     '''def row_timing_factor(mc, policies, valuation_date=VALUATION_DATE):
    """Each row's completeness over its own days relative to its year's exposure-weighted average (E22).
    Lynden-Bell by peril, mixed by each peril's share of the adjusted amount. The year total is unchanged."""
    share = mc.groupby("peril")["net_adj"].sum()
    share = share / share.sum()
    F = {k: lag_cdf_lynden_bell(g["report_lag"], (valuation_date - g["loss_date"]).dt.days) for k, g in mc.groupby("peril")}
    out = np.ones(len(policies))
    pos = pd.Series(np.arange(len(policies)), index=policies.index)
    for y, q in policies.groupby("year"):
        days = pd.date_range(f"{y}-01-01", f"{y}-12-31")
        age = (valuation_date - days).days.to_numpy()
        cday = sum(share[k] * F[k](age) for k in share.index)
        cum = np.concatenate([[0.0], np.cumsum(cday)])
        s0 = (q["period_start"] - days[0]).dt.days.to_numpy()
        e0 = (q["period_end"] - days[0]).dt.days.to_numpy() + 1
        rowc = (cum[e0] - cum[s0]) / (e0 - s0)
        avg = np.average(rowc, weights=q["exposure_clean"])
        out[pos[q.index].to_numpy()] = rowc / avg if avg > 0 else 1.0
    return out


def build_model_target(''')])
patch(R/"src"/"pipeline.py", [
    ('    T.AMOUNTS_NET = bool(cfg.get("amounts_net", False))\n    claims = load_claims(cfg["data_dir"])\n    sp = Path',
     '    T.AMOUNTS_NET = bool(cfg.get("amounts_net", False))\n    T.ROW_COMPLETENESS = bool(cfg.get("row_completeness", False))\n    claims = load_claims(cfg["data_dir"])\n    sp = Path'),
    ('    T.AMOUNTS_NET = bool(cfg.get("amounts_net", False))\n    claims = load_claims(cfg["data_dir"])\n    rows, pooled',
     '    T.AMOUNTS_NET = bool(cfg.get("amounts_net", False))\n    T.ROW_COMPLETENESS = bool(cfg.get("row_completeness", False))\n    claims = load_claims(cfg["data_dir"])\n    rows, pooled')])
patch(R/"src"/"config.py", [
    ('PART2["amounts_net"] = True                   # Oct 24: claim amounts are net of excess\n',
     'PART2["amounts_net"] = True                   # Oct 24: claim amounts are net of excess\n'
     'PART2["row_completeness"] = True              # Oct 24 (E22): completeness of each row\'s own period\n')])

patched target.py
patched pipeline.py
patched config.py


In [10]:
import sys, shutil
shutil.copy("submissions/p2_v5_net.csv", "submissions/p2_v5_net_preE22.csv")
!{sys.executable} -m pytest -q tests
!{sys.executable} -m src.pipeline --out submissions/v11_check.csv --compare submissions/v11.csv
!{sys.executable} -m src.pipeline --market part2 --validate
!{sys.executable} -m src.pipeline --market part2 --out submissions/p2_v5_net.csv --compare submissions/p2_v5_net_preE22.csv

..............................................                           [100%]
46 passed in 1.28s
14:03:17 motor: GLM, k=1.0188; Pareto refit: alpha 2.05, 119 claims above 100,000, loading 2.83; used 2.83 -> factor 1.0053; mean rate 509.23; total 101,838,084; completeness {2021: 0.999, 2022: 0.997, 2023: 0.978, 2024: 0.816} (34s)
14:03:45 buildings: GLM, k=1.0373; flat loading 2.33; mean rate 486.45; total 73,085,824; completeness {2021: 1.0, 2022: 1.0, 2023: 0.996, 2024: 0.913} (28s)
14:04:21 contents: LightGBM, 3 seeds; no loading; mean rate 107.79; total 16,181,829; completeness {2021: 1.0, 2022: 1.0, 2023: 0.998, 2024: 0.967} (36s)
14:04:22 saved submissions/v11_check.csv
compare v11.csv vs v11_check.csv: rows 626,639 of 626,639; totals 191,105,737 vs 191,105,737 (+0.000%); correlation 1.000000; rows differing by > 1%: 0.00%
14:04:28 motor: developed training table 979,968 rows
14:04:54 validate motor 2024: D2 0.0373  A/E 1.076  new 0.999  renewal 1.086
14:04:54 buildings: sum_ins

In [11]:
from pathlib import Path
d = Path.home()/"IAA"/"DECISIONS.md"; s = d.read_text()
LINE = """- **Result:** validation 2024 (same actuals as v5 before E22): pooled D² 0.1071 -> 0.1077; motor 0.0366 -> 0.0373
  with new-business A/E 0.94 -> 1.00 (as predicted by the row-level completeness); buildings 0.0688 -> 0.0684
  (new business 0.66 -> 0.69); contents 0.0538 -> 0.0539; tails within 0.01 of before. **Adopted.** Part 1 reproduced
  exactly (v11 +0.000%), 46 tests pass. Test total +1.2% (new-business rows up).
- Submissions after the 8 pm reset, expected before submitting: p2_v5a_net_noload (corrected target + E22, no
  load) 0.132-0.138; p2_v5_net (+ new business x1.3) 0.140-0.145. The pair gives the new-business optimum on the
  corrected basis.

"""
key = "### E22:"; assert key in s and "pooled D² 0.1071 -> 0.1077" not in s
i = s.find("\n\n", s.find("before the 8 pm submissions.", s.find(key))) + 2
d.write_text(s[:i] + LINE + s[i:]); print("inserted")

inserted


In [12]:
import pandas as pd, numpy as np
from pathlib import Path
D = Path.home()/"IAA"/"data_part2"/"emerging"
v5 = pd.read_csv("submissions/p2_v5_net.csv", dtype={"row_id": str})
nb = pd.concat([pd.read_parquet(D/f"test_{l}.parquet", columns=["row_id", "new_business"]) for l in ["motor","buildings","contents"]])
nb["row_id"] = nb.row_id.astype(str); m = v5.merge(nb, on="row_id", how="left", validate="one_to_one")
isnb = m.new_business.astype(bool).to_numpy()
v5a = v5.copy(); v5a.loc[isnb, "prediction"] /= 1.3
v5a.to_csv("submissions/p2_v5a_net_noload.csv", index=False)
s = np.sqrt(v5a.prediction); print("c share NB on v5a:", round(s[isnb].sum() / s.sum(), 4), "| Σsqrt v5a:", round(s.sum(), 1))

c share NB on v5a: 0.7858 | Σsqrt v5a: 3416176.6


In [14]:
from pathlib import Path
a = Path.home()/"IAA"/"AI_USE_LOG.md"; s = a.read_text()
R104_106 = """| 104 | A | AI derived that under a uniform scale k the Tweedie p=1.5 score is exactly a − b/√k − c·√k, so three submissions fix the curve, and that c can be split by segment from our own predictions, so one submission per segment gives that segment's optimum. | Expected results written before each probe; line and segment results consistent with the total. | Located the gap without changing the model: optimum about 1.45 overall, new business about 1.51. |
| 105 | E | AI read the optimum k* ≈ 1.45 as "our level is 45% too low". | The 2025 prediction was already above the emerging raw cost per exposure; validation k* equalled the total A/E (about 1). | k* is a √prediction-weighted A/E on the public rows; the test differs from training. Statement corrected in DECISIONS before any action. |
| 106 | H | Team argued that the test shift must come from outside the training data and that the competition text names it (a broker portfolio taken on in 2025). AI had called using the probe size a change of the leaderboard rule; the team pointed out that the hypothesis came from inference and each probe had its expected result written first. | Training data checked for support (broker new business in validation; channel mix and start dates of test new business): none, the book is identified only by the flag. | New-business load x1.5 adopted as a team decision (v4); size disclosed as estimated from the diagnostics. |
"""
R107 = ("| 107 | E | AI read the claim amounts as gross of excess on Oct 16 (\"some closed claims have incurred <= excess\"), "
        "and the target deducted the excess from them. The Dataset Description of both parts says paid_to_date is net of "
        "excess; AI did not check the column descriptions against that reading. The host answer of Oct 19 (\"applies to every "
        "peril\") was then read as \"deduct on every peril\", and the leaderboard signal against it (−0.0008) was set aside. | "
        "Found on Oct 24 while drafting a review prompt, after the level probes showed a test-only shortfall. Checked with "
        "data: paid = incurred on 100% of closed claims; claim frequency falls with the voluntary excess (excess 1,000 vs 0: "
        "0.47 emerging, 0.59 developed); no zero amounts. | The target deducted the excess twice in both parts (emerging x1.20 "
        "/ 1.05 / 1.19, developed x1.12 / 1.05 / 1.18 by line). Fixed for Part 2 (v5); Part 1 impact measured by a late "
        "submission: private 0.13099 -> 0.13301 (−0.0020). Lesson: every column description is checked against the target "
        "definition before modelling. |\n")
assert "| 107 |" not in s, "107 is already in the file"
if "| 106 |" in s:
    i = s.find("\n", s.find("| 106 |")) + 1
    s = s[:i] + R107 + s[i:]; print("107 added after 106")
else:
    sec = ("\n## Oct 24 — Level probes, the broker book, the excess definition\n\n"
           "| # | Type | What happened | How it was checked | Outcome |\n|---|---|---|---|---|\n" + R104_106 + R107 + "\n")
    i = s.find("## Lessons so far")
    s = s[:i] + sec.lstrip("\n") + s[i:]; print("Oct 24 table with 104-107 added")
a.write_text(s)

107 added after 106


51535

In [15]:
from pathlib import Path
a = Path.home()/"IAA"/"AI_USE_LOG.md"; s = a.read_text()
ROWS = (
"| 108 | H | Team asked a second AI to design the Part 2 model blind: a fresh chat (Claude Opus 5.5; the Fable spending "
"limit was reached, so same model family as the working session) given only the Overview, the Dataset Description "
"and raw counts. | Each point compared with our design and checked with data (notebook 26). | It defined the target "
"correctly at once (amounts already net of excess), the error found the same day, and asked whether incurred is also "
"net of recoveries (checked: no). For the broker book it did not know the direction (factor 1.0 ±15%); our diagnostics "
"estimated about 1.3, worth about 0.019 on the public score. Its offset structure was tested as E21 (+0.0002 / +0.0004, "
"not adopted). Lesson: a reader with only the documents caught what a week of data-led work had read backwards. |\n"
"| 109 | H | Team asked for an independent review of the Part 2 code, documentation and log (fresh chat, Claude Opus 5.5). "
"| Every finding triaged with data: rebutted, confirmed or accepted (DECISIONS, \"Independent review of Part 2\"). | "
"Rebutted: recoveries, 730-day truncation, IBNER, missing weather for broker addresses, history encoding, fire as a "
"weather effect. Confirmed: completeness by calendar year under-weighted late-starting new business (motor 0.777 vs "
"0.839 in 2024); fixed as E22 (motor new-business A/E 0.94 -> 1.00, pooled D² +0.0006), which moves part of the "
"new-business load from the leaderboard to the data. Accepted for the report: probe intervals by subset bootstrap, "
"three classes of assumption, premium sensitivity next to score sensitivity. |\n")
key = "| 107 |"; assert key in s and "| 108 |" not in s
i = s.find("\n", s.find(key)) + 1
a.write_text(s[:i] + ROWS + s[i:]); print("inserted")

inserted


In [16]:
import re, subprocess
from pathlib import Path
R = Path.home()/"IAA"
print("=== DECISIONS: decisions by date ===")
for line in (R/"DECISIONS.md").read_text().splitlines():
    if line.startswith("## ") or line.startswith("### "):
        print(("  " if line.startswith("### ") else "\n") + line.lstrip("# "))
print("\n=== AI_USE_LOG: entries by section and type ===")
sec = None; counts = {}
for line in (R/"AI_USE_LOG.md").read_text().splitlines():
    if line.startswith("## "): sec = line[3:]; counts[sec] = {}
    m = re.match(r"\| (\d+) \| ([AHE/]+) \|", line)
    if m and sec: counts[sec][m.group(2)] = counts[sec].get(m.group(2), 0) + 1
for k, v in counts.items():
    if v: print(f"{k}: {v}")
print("\n=== git timeline ===")
print(subprocess.run(["git", "-C", str(R), "log", "--reverse", "--date=short", "--pretty=%ad  %s"],
                     capture_output=True, text=True).stdout)

=== DECISIONS: decisions by date ===

2026-10-16 — Data structure and first target
  Data structure [02 §1]
  Data quality
  Target definition [02 §5]

2026-10-17 — Corrections, large losses, development, BI change
  Exposure correction [02 §2]
  Duplicate claims [02 §3]
  Excess on liability perils [02 §5]
  Corrected net cost per exposure [02 §6]
  Large losses [02 §7]
  Development [02 §8]
  Bodily injury severity change [02 §9]
  Motor trends [02 §10]
  Motor frequency [notebook 03]
  Motor modelling target [notebook 03, src/target.py]
  Automated tests [tests/]

2026-10-18 — Modelling plan (set before seeing any model results)
  Evaluation and validation
  Candidate models and rule
  Prediction
  How validation misses are attributed [notebook 06]

2026-10-18 — Buildings and contents adjustments [notebook 04]
  Development
  Severity trends (gross of excess)
  Frequency and weather
  Large losses
  Check: 2025-level cost per weighted exposure
  Recoveries on open claims

2026-10-19

In [19]:
import os, sys; from pathlib import Path; os.chdir(Path.home()/"IAA")
import numpy as np, pandas as pd

_a  = pd.read_csv("submissions/p2_v5a_net_noload.csv")
_b  = pd.read_csv("submissions/p2_v5_net.csv")
_v3 = pd.read_csv("submissions/p2_v3_developed.csv")
_id = _a.columns[0]
print("columns:", list(_a.columns), "| rows:", len(_a), len(_b), len(_v3))
assert list(_a.columns) == list(_b.columns) == list(_v3.columns)
assert len(_a) == len(_b) == len(_v3)
assert (_a[_id].values == _b[_id].values).all() and (_a[_id].values == _v3[_id].values).all(), "id order differs"
for _d, _n in [(_a, "v5a"), (_b, "v5")]:
    assert _d.prediction.notna().all() and (_d.prediction > 0).all(), _n

_r = (_b.prediction / _a.prediction).round(6)
print(_r.value_counts())
assert set(_r.unique()) <= {1.0, 1.3}, "unexpected ratio"
print("OK: both files ready to submit")

columns: ['row_id', 'prediction'] | rows: 209063 209063 209063
prediction
1.3    158980
1.0     50083
Name: count, dtype: int64
OK: both files ready to submit


In [18]:
import os, sys; from pathlib import Path; os.chdir(Path.home()/"IAA")
import numpy as np, pandas as pd

_a  = pd.read_csv("submissions/p2_v5a_net_noload.csv")
_b  = pd.read_csv("submissions/p2_v5_net.csv")
_v3 = pd.read_csv("submissions/p2_v3_developed.csv")
nb_mask = (_b.prediction / _a.prediction).round(6) == 1.3
sq5a = np.sqrt(_a.prediction)
print(f"NB rows {nb_mask.mean():.3f} | Σ√v5a {sq5a.sum():.1f} | NB share {sq5a[nb_mask].sum()/sq5a.sum():.4f}")

k = 1.3
c_tot = 0.47296 * sq5a.sum() / np.sqrt(_v3.prediction).sum()
cN = c_tot * sq5a[nb_mask].sum() / sq5a.sum()
gain = lambda kN: cN * (kN * (1 - k**-0.5) - (k**0.5 - 1))
lo, hi = gain(1.25), gain(1.35)
print(f"c total {c_tot:.4f} | c NB {cN:.4f}")
for kN in [k**0.5, 1.2, 1.25, 1.3, 1.35, 1.45]:
    print(f"  NB optimum x{kN:.3f} -> S_v5 - S_v5a = {gain(kN):+.5f}")

txt = f"""### 2026-10-24 — v5a / v5 pre-registered expectations (before 20:00 ET reset)
- p2_v5a_net_noload (amounts_net + E22, no load): expected public 0.132–0.138
- p2_v5_net (same + NB x1.3): expected public 0.140–0.145
- Decision rule: NB optimum on corrected basis kN = bN/cN (cN = {cN:.4f}). If 1.25 <= kN <= 1.35, keep v5 as final, no fine-tuning (0.05 of load = ±0.0002).
  Otherwise build v6 with nb_load = kN.
- Same rule in score terms: keep v5 iff {lo:.5f} <= S_v5 - S_v5a <= {hi:.5f}. Gain <= 0 means kN <= sqrt(1.3) = 1.140.
"""
p = Path("DECISIONS.md"); t = p.read_text(encoding="utf-8")
assert "v5a / v5 pre-registered expectations" not in t, "already recorded"
anchor = "## Assumptions register"
assert t.count(anchor) == 1
p.write_text(t.replace(anchor, txt + "\n" + anchor), encoding="utf-8")
print("recorded")

NB rows 0.760 | Σ√v5a 3416176.6 | NB share 0.7858
c total 0.5102 | c NB 0.4009
  NB optimum x1.140 -> S_v5 - S_v5a = -0.00000
  NB optimum x1.200 -> S_v5 - S_v5a = +0.00295
  NB optimum x1.250 -> S_v5 - S_v5a = +0.00541
  NB optimum x1.300 -> S_v5 - S_v5a = +0.00788
  NB optimum x1.350 -> S_v5 - S_v5a = +0.01034
  NB optimum x1.450 -> S_v5 - S_v5a = +0.01527
recorded


In [20]:
import os, sys; from pathlib import Path; os.chdir(Path.home()/"IAA")
import numpy as np, pandas as pd

S_a, S_5, k = 0.13541, 0.14287, 1.3
_v3 = pd.read_csv("submissions/p2_v3_developed.csv")
c_tot = 0.47296 * 3416176.6 / np.sqrt(_v3.prediction).sum()
cN = c_tot * 0.7858
gain = S_5 - S_a
bN = -(gain + cN * (k**0.5 - 1)) / (k**-0.5 - 1)
kN = bN / cN
best = S_a + bN + cN - 2 * np.sqrt(bN * cN)
print(f"gain {gain:+.5f} | c NB {cN:.4f} | b NB {bN:.4f} | NB optimum x{kN:.3f} | expected at optimum {best:.5f}")
assert 1.25 <= kN <= 1.35

txt = f"""### 2026-10-24 20:00 — v5a / v5 results and final choice
- p2_v5a_net_noload: public {S_a:.5f} (expected 0.132–0.138, inside)
- p2_v5_net (NB x1.3): public {S_5:.5f} (expected 0.140–0.145, inside); +{S_5-0.14133:.5f} over v4
- S_v5 - S_v5a = {gain:+.5f}, inside the pre-registered window. NB optimum on the corrected basis x{kN:.3f}; expected public at the optimum {best:.5f} (v5 is {best-S_5:.5f} below it).
- Decision (rule set before the scores): **p2_v5_net is the Part 2 final.** No fine-tuning.
"""
p = Path("DECISIONS.md"); t = p.read_text(encoding="utf-8")
assert "v5a / v5 results and final choice" not in t, "already recorded"
anchor = "## Assumptions register"
assert t.count(anchor) == 1
p.write_text(t.replace(anchor, txt + "\n" + anchor), encoding="utf-8")
print("recorded")

gain +0.00746 | c NB 0.4009 | b NB 0.5178 | NB optimum x1.292 | expected at optimum 0.14287
recorded


In [22]:
import os, sys; from pathlib import Path; os.chdir(Path.home()/"IAA")
!{sys.executable} tools/exp_blackbox_p2b.py --help

usage: exp_blackbox_p2b.py [-h] [--out OUT] [--compare COMPARE] [--validate]
                           [--lines LINES [LINES ...]] [--data DATA]

Experiment E9-P2b: the black box with the same data as ours (developed + emerging training rows). Benchmark only.

E9-P2 (Oct 23) trained the black box on emerging data only, while v3 (Oct 24, E17) learns relativities from the
developed market too. A fair comparison needs the same data, so here the black box is LightGBM (E9 settings) fitted
on developed 2021-24 + emerging training rows with a "market" factor, every feature the two markets share, no peril
split; its level is set on the emerging training rows (one factor), as ours is.

Outputs
  --validate   fit (developed + emerging 2023) -> validate emerging 2024, all perils: ours (current PART2 = v3) vs the
               black box on emerging only (E9-P2) vs the black box on combined data; D2, A/E, lowest / highest decile,
               new-business A/E, seed spread; pooled D2.
  --out FI

In [23]:
import os, sys; from pathlib import Path; os.chdir(Path.home()/"IAA")
import re
src = Path("tools/exp_blackbox_p2b.py").read_text(encoding="utf-8").splitlines()
pat = re.compile(r"import|config|PART2|target|amounts_net|row_completeness|nb_load|completeness|excess|pipeline", re.I)
for i, line in enumerate(src, 1):
    if pat.search(line):
        print(f"{i:4d}: {line}")

   9:   --validate   fit (developed + emerging 2023) -> validate emerging 2024, all perils: ours (current PART2 = v3) vs the
  17: import argparse
  18: import copy
  19: import sys
  20: import time
  21: from pathlib import Path
  23: import numpy as np
  24: import pandas as pd
  28: import exp_blackbox_p2 as E  # noqa: E402  (also patches the pipeline's GBM fit for one-year folds)
  29: import src.pipeline as P  # noqa: E402
  30: from src.config import PART1, PART2  # noqa: E402
  31: from src.evaluate import evaluate  # noqa: E402
  32: from src.target import load_claims  # noqa: E402
  53:     cols = f + ["target", "weight", "policy_id"]
  59:     lvl = em.target.sum() / (m.predict(Xe) * em.weight).sum()
  73:     ours, bb = copy.deepcopy(PART2), E.blackbox_config()
  99:             act = (va.target_all if "target_all" in va else va.target).to_numpy()
 119:     cfg = E.blackbox_config()
 149:     ap.add_argument("--lines", nargs="+", default=list(PART2["lines"]))


In [24]:
import os, sys; from pathlib import Path; os.chdir(Path.home()/"IAA")
import re
p2b = Path("tools/exp_blackbox_p2b.py").read_text(encoding="utf-8").splitlines()
print("=== exp_blackbox_p2b.py 40-130 ===")
for i in range(39, min(130, len(p2b))):
    print(f"{i+1:4d}: {p2b[i]}")

p2 = Path("tools/exp_blackbox_p2.py").read_text(encoding="utf-8").splitlines()
start = next(i for i, l in enumerate(p2) if re.match(r"def blackbox_config", l))
end = next((i for i in range(start + 1, len(p2)) if re.match(r"(def |class |\S)", p2[i])), len(p2))
print("\n=== exp_blackbox_p2.py blackbox_config ===")
for i in range(start, end):
    print(f"{i+1:4d}: {p2[i]}")

=== exp_blackbox_p2b.py 40-130 ===
  40:         c1 = load_claims(PART1["data_dir"])
  41:         d1 = P.build_line(line, PART1, c1)
  42:         t = d1["tr"].merge(d1["p"][["row_id", "policy_id"]], on="row_id", how="left")
  43:         age, band = P.AGE_COLS[line]
  44:         if age not in t:
  45:             t = t.merge(d1["p"][["row_id", age]], on="row_id", how="left")
  46:         t[band] = pd.cut(t[age], P.AGE_BANDS[0], labels=P.AGE_BANDS[1]).astype(str)
  47:         _DEV[line] = t
  48:     return _DEV[line]
  49: 
  50: 
  51: def fit_combined(em, dev, feats, params, seed):
  52:     f = [c for c in feats if c in dev.columns]
  53:     cols = f + ["target", "weight", "policy_id"]
  54:     both = pd.concat([dev[cols].assign(market="developed", policy_id=lambda x: "D" + x.policy_id.astype(str)),
  55:                       em[cols].assign(market="emerging", policy_id=lambda x: "E" + x.policy_id.astype(str))],
  56:                      ignore_index=True)
  57:     m, cats

In [25]:
import os, sys; from pathlib import Path; os.chdir(Path.home()/"IAA")
import re
pat = re.compile(r"nb_load|PART1|amounts_net|row_completeness|developed|dev_cfg|dev_market", re.I)
for f in sorted(Path("src").glob("*.py")):
    for i, line in enumerate(f.read_text(encoding="utf-8").splitlines(), 1):
        if pat.search(line):
            print(f"{f.name}:{i:4d}: {line.rstrip()}")

checks.py:  18:   python -m src.checks --out results/checks_part1
checks.py:  19:   python -m src.checks --out results/checks_part2 --data <part2 data dir> --reference results/checks_part1/schema.json
checks.py: 257:     ap.add_argument("--out", default="results/checks_part1")
config.py:   3: PART1 reproduces the adopted model (v11, DECISIONS.md Oct 21). For Part 2, copy PART1, change the values
config.py:  27: PART1 = {
config.py:  28:     "name": "part1",
config.py:  64: PART2 = copy.deepcopy(PART1)
config.py:  75:     "interactions": {},                        # Part 1 pairs were selected on developed data; re-select before use
config.py:  81: PART2["adj"]["motor"]["reform"] = {}           # BI reform 0.83 is a developed-market change (Oct 23)
config.py:  90: PART2["peril_split"] = {"buildings": {"peril": "fire", "rate": "developed"}}   # or a number per exposure
config.py: 101: # Oct 24 (E17, notebook 23): developed data as training data (v3)
config.py: 103: PART2["developed_traini

In [26]:
import os, sys, re, py_compile; from pathlib import Path; os.chdir(Path.home()/"IAA")

p = Path("tools/exp_blackbox_p2b.py"); t = p.read_text(encoding="utf-8")
assert "T.AMOUNTS_NET" not in t, "already patched"
assert re.search(r"\bT\b", t) is None, "name T already used"

imp = "from src.target import load_claims  # noqa: E402"
assert t.count(imp) == 1
t = t.replace(imp, imp + "\nimport src.target as T  # noqa: E402")

old_v = "    ours, bb = copy.deepcopy(PART2), E.blackbox_config()\n"
assert t.count(old_v) == 1
t = t.replace(old_v, old_v + '    T.AMOUNTS_NET, T.ROW_COMPLETENESS = bool(ours.get("amounts_net")), bool(ours.get("row_completeness"))  # as pipeline.validate\n')

old_r = "    cfg = E.blackbox_config()\n"
assert t.count(old_r) == 1
t = t.replace(old_r, old_r + '    T.AMOUNTS_NET, T.ROW_COMPLETENESS = bool(cfg.get("amounts_net")), bool(cfg.get("row_completeness"))  # as pipeline.run\n')

old_l = '"ours (v3)"'
assert t.count(old_l) == 1
t = t.replace(old_l, '"ours (current PART2)"')

p.write_text(t, encoding="utf-8")
py_compile.compile(str(p), doraise=True)
print("patched + compiles")

for i, line in enumerate(Path("tools/exp_blackbox_p2.py").read_text(encoding="utf-8").splitlines(), 1):
    if re.search(r"AMOUNTS_NET|ROW_COMPLETENESS", line):
        print(f"exp_blackbox_p2.py:{i}: {line}")

patched + compiles


In [27]:
!{sys.executable} tools/exp_blackbox_p2b.py --validate

20:09:11 motor: developed training table 979,968 rows
20:09:35 motor ours (current PART2): D2 0.0373 (29s)
20:09:43 motor black box, emerging only: D2 0.0174 (8s)
20:10:04 motor combined black box seed 42: 55 features, 68 trees
20:10:25 motor combined black box seed 43: 55 features, 68 trees
20:10:45 motor combined black box seed 44: 55 features, 58 trees
20:10:45 motor black box, combined: D2 0.0390 (62s)
20:10:49 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
20:10:49 buildings: fire priced separately at 97.05 per exposure (developed)
20:10:49 buildings: own GLM for ['flood']; main GLM on the other perils
20:10:54 buildings: developed training table 707,726 rows
20:11:01 buildings ours (current PART2): D2 0.0684 (13s)
20:11:01 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
20:11:02 buildings: own GLM for ['flood']; main GLM on the other perils
20:11:06 buildings black box, emerging only: D2 -0.0619 (5s)
20:11:06 buildings: sum_insur

In [28]:
import os, sys; from pathlib import Path; os.chdir(Path.home()/"IAA")

dec = """### 2026-10-24 20:15 — E9-P2b rerun on the corrected target (notebook 26)
- Patch: tools/exp_blackbox_p2b.py calls build_line directly and did not set T.AMOUNTS_NET / T.ROW_COMPLETENESS (set only in pipeline.run / validate), so after the Oct 24 fixes it would have compared all models, and the actuals, on the old target. It now sets them from its config, as the pipeline does.
- Validation (fit 2023 + developed 2021-24 -> 2024, all perils), D2 ours vs combined black box: motor 0.0373 vs 0.0390 (+0.0017; black-box seed spread 0.08); contents 0.0539 vs 0.0396; buildings 0.0684 vs -0.0383 (black-box A/E 0.53, level not comparable, left out of the conclusion); pooled 0.1077 vs 0.0832.
- Rule set before the run: black box - ours >= +0.005 -> shape gap is real; < +0.002 -> stop. Motor +0.0017, contents negative -> **GBM line closed.** The public gap to the top teams (0.154-0.168) is not explained by model class on our data.
- Signals for follow-up, validation only, credibility not yet checked: lowest-decile A/E below the line average (motor 0.79, contents 0.74, buildings 0.35) -> model may be too flat; new-business A/E by line in opposite directions (buildings 0.69, contents 1.35, motor 1.00) while the test load is one x1.3 for all lines.
"""
p = Path("DECISIONS.md"); t = p.read_text(encoding="utf-8")
assert "E9-P2b rerun on the corrected target" not in t, "already recorded"
anchor = "## Assumptions register"
assert t.count(anchor) == 1
p.write_text(t.replace(anchor, dec + "\n" + anchor), encoding="utf-8")

log = ("| 110 | E | The Oct 24 target fixes were added as module switches set only in `pipeline.run` / `validate`; "
       "the side script `exp_blackbox_p2b.py`, which calls `build_line` directly, was not updated, so it would have compared models on the old target. "
       "| Found before running, by reading where the switches are set (notebook 26). "
       "| Script patched to set the switches from its config; rerun on the corrected target (DECISIONS, E9-P2b rerun): no black-box gain. "
       "Lesson: a global switch is checked in every entry point, not only the pipeline. |\n")
q = Path("AI_USE_LOG.md"); lines = q.read_text(encoding="utf-8").splitlines(keepends=True)
assert not any(l.startswith("| 110 |") for l in lines), "110 already exists"
idx = [i for i, l in enumerate(lines) if l.startswith("| 109 |")]
assert len(idx) == 1
lines.insert(idx[0] + 1, log)
q.write_text("".join(lines), encoding="utf-8")
print("recorded")

recorded
